In [5]:
%load_ext autoreload
%autoreload 2
# you can check out the the documentation for the rest of the autoreaload modes
# by apending a question mark to %autoreload, like this:
# %autoreload?


from IPython.lib import deepreload
import imp
def my_reload(modules=[], deep=False):
    if not isinstance(modules, list):
        modules = [modules]
    if deep:
        rl_func = deepreload.reload
    else:
        rl_func = imp.reload
    
    for m in modules:
        try:
            rl_func(m)
        except:
            print(f"THIS IS A COMMON ISSUE: some modules in {m} are not work")

import os
from pathlib import Path
from datasets import load_dataset

import nest_asyncio
nest_asyncio.apply()

main_dir = Path("/home/lyb")
data_dir = main_dir / "RAG/data/eli5_data"

/tmp/ipykernel_157857/1590991188.py:9: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp


### Data Preparation (QA & Corpus) 

In [2]:
def load_eli5_dataset(save_path):
    # set file path
    file_path = "MarkrAI/eli5_sample_autorag"

    # load dataset
    corpus_dataset = load_dataset(file_path, "corpus")['train'].to_pandas()
    qa_train_dataset = load_dataset(file_path, "qa")['train'].to_pandas()
    qa_test_dataset = load_dataset(file_path, "qa")['test'].to_pandas()

    # save data
    if os.path.exists(os.path.join(save_path, "corpus.parquet")) is True:
        raise ValueError("corpus.parquet already exists")
    if os.path.exists(os.path.join(save_path, "qa.parquet")) is True:
        raise ValueError("qa.parquet already exists")
    corpus_dataset.to_parquet(os.path.join(save_path, "corpus.parquet"))
    qa_train_dataset.to_parquet(os.path.join(save_path, "qa_train.parquet"))
    qa_test_dataset.to_parquet(os.path.join(save_path, "qa_test.parquet"))


load_eli5_dataset(data_dir)

Some datasets params were ignored: ['splits']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some datasets params were ignored: ['splits']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Generating train split:   0%|          | 0/600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

Some datasets params were ignored: ['splits']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


ValueError: corpus.parquet already exists

In [5]:
import pandas as pd

qa_df = pd.read_parquet(data_dir / 'qa_train.parquet')
sample_qa_df = qa_df.sample(20, random_state=42) # In this sample code, we will only optimize pipeline with 20 samples just for testing.
sample_qa_df.reset_index(drop=True, inplace=True)
sample_qa_df.to_parquet(data_dir / 'qa_sample.parquet')

from itertools import chain
from autorag.utils.util import to_list
# We drop unused corpus dataframe for faster inference.
corpus_df = pd.read_parquet(data_dir / 'corpus.parquet')
target_retrieval_gts = list(chain.from_iterable(to_list(sample_qa_df["retrieval_gt"].tolist())))
target_retrieval_gts = list(chain.from_iterable(target_retrieval_gts))
sample_corpus_df = corpus_df[corpus_df["doc_id"].isin(target_retrieval_gts)]
sample_corpus_df.reset_index(drop=True, inplace=True)
sample_corpus_df.to_parquet(data_dir / 'corpus_sample.parquet')

[12/20/24 06:10:17] INFO     [__init__.py:100] >> You are using API version of AutoRAG.To use local ]8;id=567134;file:///home/lyb/RAG/AutoRAG/autorag/__init__.py\__init__.py]8;;\:]8;id=746580;file:///home/lyb/RAG/AutoRAG/autorag/__init__.py#100\100]8;;\
                             version, run pip install 'AutoRAG[gpu]'                                               

                    INFO     [__init__.py:128] >> You are using API version of AutoRAG.To use local ]8;id=406636;file:///home/lyb/RAG/AutoRAG/autorag/__init__.py\__init__.py]8;;\:]8;id=402932;file:///home/lyb/RAG/AutoRAG/autorag/__init__.py#128\128]8;;\
                             version, run pip install 'AutoRAG[gpu]'                                               

### Evaluate RAG Generation

In [ ]:
import pandas as pd
from autorag.schema.metricinput import MetricInput
from autorag.evaluation import evaluate_generation

# Load QA dataset
qa_df = pd.read_parquet("qa.parquet", engine="pyarrow")

# Prepare MetricInput list
metric_inputs = [
    MetricInput(query=row["query"], generation_gt=row["generation_gt"])
    for _, row in qa_df.iterrows()
]

# Define custom generation function with decorator
@evaluate_generation(
    metric_inputs=metric_inputs,
    metrics=["bleu", "meteor", "rouge"]
)
def custom_generation(queries):
    # Implement your generation logic
    return generated_texts, [[1, 30]] * len(generated_texts), [[-1, -1.3]] * len(generated_texts)

# Evaluate generation performance
generation_result_df = custom_generation(qa_df["query"].tolist())

In [6]:
from autorag.evaluator import Evaluator
evaluator = Evaluator(qa_data_path=(data_dir/'qa_sample.parquet').as_posix(), corpus_data_path=(data_dir/'corpus.parquet').as_posix(),
                      project_dir='./eli5_runs')

In [8]:
evaluator.start_trial('./candidate_config/ollama_config_0.yaml', skip_validation=True)

[01/30/25 12:27:38] INFO     [evaluator.py:228] >> Embedding BM25 corpus...                        ]8;id=406658;file:///home/lyb/RAG/AutoRAG/autorag/evaluator.py\evaluator.py]8;;\:]8;id=110163;file:///home/lyb/RAG/AutoRAG/autorag/evaluator.py#228\228]8;;\

[01/30/25 12:27:55] INFO     [evaluator.py:248] >> BM25 corpus embedding complete.                 ]8;id=33412;file:///home/lyb/RAG/AutoRAG/autorag/evaluator.py\evaluator.py]8;;\:]8;id=915499;file:///home/lyb/RAG/AutoRAG/autorag/evaluator.py#248\248]8;;\

Output()

                    INFO     [evaluator.py:205] >> Running node line retrieve_node_line...         ]8;id=853155;file:///home/lyb/RAG/AutoRAG/autorag/evaluator.py\evaluator.py]8;;\:]8;id=351646;file:///home/lyb/RAG/AutoRAG/autorag/evaluator.py#205\205]8;;\

Running node line in ./eli5_runs/3/retrieve_node_line...

...

task_eval: 0

                    INFO     [node.py:55] >> Running node generator...                                   ]8;id=873051;file:///home/lyb/RAG/AutoRAG/autorag/schema/node.py\node.py]8;;\:]8;id=960382;file:///home/lyb/RAG/AutoRAG/autorag/schema/node.py#55\55]8;;\

                    INFO     [base.py:19] >> Initialize generator node - LlamaIndexLLM                   ]8;id=894403;file:///home/lyb/RAG/AutoRAG/autorag/nodes/generator/base.py\base.py]8;;\:]8;id=277309;file:///home/lyb/RAG/AutoRAG/autorag/nodes/generator/base.py#19\19]8;;\

                    INFO     [base.py:26] >> Running generator node - LlamaIndexLLM module...            ]8;id=195209;file:///home/lyb/RAG/AutoRAG/autorag/nodes/generator/base.py\base.py]8;;\:]8;id=867756;file:///home/lyb/RAG/AutoRAG/autorag/nodes/generator/base.py#26\26]8;;\

AssertionError: previous_result must contain prompts column.

In [7]:
from autorag.deploy import extract_best_config
extract_best_config(trial_path='./eli5/0', output_path='./eli5/0/best.yaml')

{'node_lines': [{'node_line_name': 'retrieve_node_line',
   'nodes': [{'node_type': 'retrieval',
     'strategy': {'metrics': ['retrieval_f1',
       'retrieval_recall',
       'retrieval_precision']},
     'modules': [{'module_type': 'HybridCC',
       'top_k': 3,
       'target_modules': ('VectorDB', 'BM25'),
       'weights': (0.3, 0.7),
       'weight': 0.0,
       'target_module_params': ({'top_k': 3, 'vectordb': 'chroma_mpnet'},
        {'top_k': 3})}]}]},
  {'node_line_name': 'post_retrieve_node_line',
   'nodes': [{'node_type': 'prompt_maker',
     'strategy': {'metrics': ['meteor', 'rouge', 'bert_score']},
     'modules': [{'module_type': 'Fstring',
       'prompt': 'Read the passages and answer the given question. \n Question: {query} \n Passage: {retrieved_contents} \n Answer : '}]},
    {'node_type': 'generator',
     'strategy': {'metrics': ['meteor', 'rouge', 'bert_score']},
     'modules': [{'module_type': 'LlamaIndexLLM',
       'llm': 'ollama',
       'model': 'llama3'

In [2]:
from autorag.deploy import Runner
config_path = "./candidate_config/ollama_config_0.yaml"
runner = Runner.from_yaml(config_path, project_dir='./eli5')
runner.run('I am not good. How about you?')

[01/30/25 11:29:29] INFO     [config.py:58] >> PyTorch version 2.5.1 available.                        ]8;id=886746;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/datasets/config.py\config.py]8;;\:]8;id=258743;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/datasets/config.py#58\58]8;;\

/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in DeployedModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fiel

[01/30/25 11:29:31] INFO     [_client.py:1026] >> HTTP Request: GET                                 ]8;id=35854;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=441853;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/httpx/_client.py#1026\1026]8;;\
                             https://api.gradio.app/gradio-messaging/en "HTTP/1.1 200 OK"                          

HybridRetrieval Init!!!!
target_modules: ('bm25', 'vectordb') [{'top_k': 3}, {'top_k': 3, 'vectordb': 'default_vectordb'}]


[01/30/25 11:29:33] INFO     [base.py:18] >> Initialize retrieval node - HybridCC                        ]8;id=789922;file:///home/lyb/RAG/AutoRAG/autorag/nodes/retrieval/base.py\base.py]8;;\:]8;id=976313;file:///home/lyb/RAG/AutoRAG/autorag/nodes/retrieval/base.py#18\18]8;;\

                    INFO     [base.py:18] >> Initialize retrieval node - BM25                            ]8;id=918355;file:///home/lyb/RAG/AutoRAG/autorag/nodes/retrieval/base.py\base.py]8;;\:]8;id=537990;file:///home/lyb/RAG/AutoRAG/autorag/nodes/retrieval/base.py#18\18]8;;\

AssertionError: bm25_path ./eli5/resources/bm25_porter_stemmer.pkl does not exist. Please ingest first.

In [5]:
!autorag run_web --yaml_path ./eli5/0/best.yaml --project_dir ./eli5


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[12/21/24 16:19:56] INFO     [config.py:58] >> PyTorch version      ]8;id=239230;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/datasets/config.py\config.py]8;;\:]8;id=803982;file:///home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/datasets/config.py#58\58]8;;\
                             2.5.1 available.                                   
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in DeployedModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.conda/envs/autorag/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceLLM has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/lyb/.c

### 5 Dataset Evaluation

In [35]:
import pandas as pd
from pathlib import Path

# Define file path
trial_path = Path("/home/lyb/RAG/AutoRAG/experiments/test_runner")
csv_file_path = trial_path / 'ragas/output/final_results.csv'

# Load CSV efficiently using Pandas (only necessary column)
df = pd.read_csv(csv_file_path, usecols=["qid", "meteor", "rouge", "bert_score"])

# Define segment size
segment_size = 100

# Create segment labels (0 for first 100 rows, 1 for next 100, etc.)
df["segment"] = df.index // segment_size

# Compute the average bert_score for each 100-row segment
average_scores = df.groupby("segment")[["meteor", "rouge", "bert_score"]].mean()

# Print results
print(average_scores)

dataset_names = ["2WikiMultihopQA", "hotpot", "musique", "NQ", "trivalQA"]
# add a new column to the dataframe that contains the name of each segment
average_scores["dataset_name"] = dataset_names[: len(average_scores)]


average_scores

           meteor     rouge  bert_score
segment                                
0        0.133940  0.062386    0.795542
1        0.122588  0.055753    0.802504
2        0.119668  0.052208    0.799217
3        0.084292  0.073094    0.799388
4        0.077717  0.036183    0.787381


,meteor,rouge,bert_score,dataset_name
segment,,,,
0,0.133940,0.062386,0.795542,2WikiMultihopQA
1,0.122588,0.055753,0.802504,hotpot
2,0.119668,0.052208,0.799217,musique
3,0.084292,0.073094,0.799388,NQ
4,0.077717,0.036183,0.787381,trivalQA
